# Response generation

[Fine-tuning Llama 2 with LoRA](https://deci.ai/blog/fine-tune-llama-2-with-lora-for-question-answering/)

# Fine-tuning
new_model will be the name of your fine-tuned model (saved)

In [14]:
import os, torch, logging
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

from flash_attn import flash_attn_func
import accelerate

import wandb
from huggingface_hub import login
import os

In [15]:

%load_ext dotenv
%dotenv
login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=True)

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (cache).
Your token has been saved to /home/user/.cache/huggingface/token
Login successful


In [16]:
# Dataset
data_name = "mlabonne/guanaco-llama2-1k"
training_data = load_dataset(data_name, split="train")

# Model and tokenizer names
base_model_name = "meta-llama/Llama-2-7b-chat-hf"
refined_model = "llama-2-7b-testing"

# Tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

# Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

# Model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    attn_implementation="flash_attention_2",
    quantization_config=quant_config,
    device_map={"": 0},
    use_cache=False,
    low_cpu_mem_usage=True
)

base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
# Model and tokenizer names
base_model_name = "meta-llama/Llama-2-7b-chat-hf"
refined_model = "llama-2-7b-testing"

# Tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

In [18]:
from transformers import pipeline

pipe = pipeline(task="text-generation", model=base_model, tokenizer=llama_tokenizer)
messages = [
    {
        "role": "system",
        "content": "rewrite with anger emotions",
    },
    {"role": "user", "content": "Say , Jim , how about going for a few beers after dinner ? "},
]
print(pipe(messages, max_new_tokens=128)[0]['generated_text'][-1])  # Print the assistant's response

{'role': 'assistant', 'content': '  "JIM! ARE YOU KIDDING ME?! AFTER THIS LONG DAY OF WORK, YOU WANT TO GO FOR A FEW BEERS?! I\'VE HAD A BETTER IDEA! LET\'S GO GRAB A BOTTLE OF RAGE AND STORM OFF TO THE MOST CROWDED BAR WE CAN FIND, AND SHOUT OUR ANGER TO THE ROOFTOPS! THAT\'S WHAT I\'M TALKING ABOUT! F'}
